# CareLoop Clinics – Part 1: Python & Data Foundations
**Tasks 1.1 – 1.6: Ingest, Validate, Clean, Enrich and Export**

This notebook implements Part 1 of the CareLoop Clinics project in Google Colab–compatible Python.
- Original datasets are read directly without modification.
- Every task follows the exact assignment specifications and acceptance criteria.

## Environment & Path Setup
Resolve input dataset paths (`appointment.csv` / `appointments.csv` and `clinics.csv`) and output directories so the notebook runs seamlessly both locally and in Google Colab.

In [1]:
import os

def resolve_input_file(candidates):
    """Return the first existing file path from candidate locations."""
    search_roots = [".", "..", "data/raw", "../data/raw"]
    for root in search_roots:
        for name in candidates:
            path = os.path.join(root, name)
            if os.path.isfile(path):
                return path
    raise FileNotFoundError(f"Could not find any of {candidates} in {search_roots}")

PROJECT_ROOT = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
PROCESSED_DIR = os.path.join(PROJECT_ROOT, "data", "processed")
OUTPUTS_DIR = os.path.join(PROJECT_ROOT, "outputs")
SQL_DIR = os.path.join(PROJECT_ROOT, "sql")

for d in (PROCESSED_DIR, OUTPUTS_DIR, SQL_DIR):
    os.makedirs(d, exist_ok=True)

APPOINTMENTS_PATH = resolve_input_file(["appointments.csv", "appointment.csv"])
CLINICS_PATH = resolve_input_file(["clinics.csv"])

print("Appointments input:", APPOINTMENTS_PATH)
print("Clinics input     :", CLINICS_PATH)
print("Processed dir     :", PROCESSED_DIR)
print("Outputs dir       :", OUTPUTS_DIR)
print("SQL dir           :", SQL_DIR)

Appointments input: .\appointment.csv
Clinics input     : .\clinics.csv
Processed dir     : .\data\processed
Outputs dir       : .\outputs
SQL dir           : .\sql


## Task 1.1 – Read the File the Foundations Way (5 Marks)
Use ONLY Python's built-in `csv` and `json` modules (no Pandas). Parse numeric fields (`age`, `fee_inr`, `prior_no_shows` as `int`, `distance_km` as `float`) without logical validation, separate accepted vs. rejected rows, and save `rejected_rows.json`.

In [2]:
import csv
import json

def parse_row(row):
    """Convert numeric string fields in an appointment row to int/float."""
    parsed = dict(row)
    parsed["age"] = int(row["age"])
    parsed["fee_inr"] = int(row["fee_inr"])
    parsed["prior_no_shows"] = int(row["prior_no_shows"])
    parsed["distance_km"] = float(row["distance_km"])
    return parsed

accepted = []
rejected = []

with open(APPOINTMENTS_PATH, newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    header = reader.fieldnames
    rows = list(reader)

print("CSV Header:", header)
print("Total number of rows:", len(rows))

for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError:
        rejected.append(row)

rejected_path = os.path.join(OUTPUTS_DIR, "rejected_rows.json")
with open(rejected_path, "w", encoding="utf-8") as file:
    json.dump(rejected, file, indent=2)

blank_age_count = sum(1 for r in rejected if r["age"].strip() == "")
blank_dist_count = sum(1 for r in rejected if r["distance_km"].strip() == "")

print("Accepted rows:", len(accepted))
print("Rejected rows:", len(rejected))
print("  - Rejected due to blank age:", blank_age_count)
print("  - Rejected due to blank distance_km:", blank_dist_count)
print("Saved rejected rows to:", rejected_path)

CSV Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Total number of rows: 1020
Accepted rows: 988
Rejected rows: 32
  - Rejected due to blank age: 12
  - Rejected due to blank distance_km: 20
Saved rejected rows to: .\outputs\rejected_rows.json


## Task 1.2 – Inspect and Clean, Pass 1 (5 Marks)
Start fresh from the original appointments CSV using Pandas. Inspect shape, info, missing values, and duplicates, then clean in the exact required order: drop exact duplicates → clean `booking_channel` → convert dates → drop missing `appt_date` and `status`.

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv(APPOINTMENTS_PATH)

print("1. df.shape:", df.shape)
print("\n2. df.info():")
df.info()
print("\n3. df.isnull().sum():")
print(df.isnull().sum())
print("\n4. df.duplicated().sum():", df.duplicated().sum())

# STEP A - Drop exact duplicate rows
df = df.drop_duplicates()
print("\nAfter STEP A (drop_duplicates) row count:", len(df))

# STEP B - Clean booking_channel
before_channels = df["booking_channel"].nunique()
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
after_channels = df["booking_channel"].nunique()
print(f"\nSTEP B (booking_channel): {before_channels} distinct values before -> {after_channels} distinct values after")
print(df["booking_channel"].value_counts())

# STEP C - Convert dates
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("\nSTEP C: Unreadable/missing appt_date after conversion:", df["appt_date"].isna().sum())

# STEP D - Drop missing appt_date, then missing status
df = df.dropna(subset=["appt_date"])
print("After dropping missing appt_date row count:", len(df))

df = df.dropna(subset=["status"])
print("After dropping missing status row count:", len(df))

1. df.shape: (1020, 14)

2. df.info():
<class 'pandas.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   str    
 1   patient_id       1020 non-null   str    
 2   clinic_id        1020 non-null   str    
 3   booked_on        1020 non-null   str    
 4   appt_date        1019 non-null   str    
 5   age              1008 non-null   float64
 6   gender           1020 non-null   str    
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   str    
 10  visit_type       1020 non-null   str    
 11  reminder_sent    1020 non-null   str    
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   str    
dtypes: float64(2), int64(2), str(10)
memory usage: 111.7 KB

3. df.isnull().sum():
appointment_id      0
patient_id          0
c

## Task 1.3 – Clean, Pass 2: Validity, Gaps and Outliers (5 Marks)
Continue with the DataFrame from Task 1.2: replace invalid ages (`< 1` or `> 100`) with `NaN`, fill missing `age` and `distance_km` with their medians, cap outliers in `distance_km` and `fee_inr` using IQR fences and `np.clip`, and apply final rounding.

In [4]:
# STEP A - Validate age (< 1 or > 100 -> NaN)
invalid_age_mask = (df["age"] < 1) | (df["age"] > 100)
invalid_age_count = int(invalid_age_mask.sum())
df.loc[invalid_age_mask, "age"] = np.nan
missing_age_after = int(df["age"].isna().sum())

print(f"STEP A: Invalid ages replaced with NaN: {invalid_age_count}")
print(f"STEP A: Total missing age values after replacement: {missing_age_after}")

# STEP B - Fill missing values with medians
median_age = df["age"].median()
df["age"] = df["age"].fillna(median_age).round().astype(int)

median_distance = df["distance_km"].median()
df["distance_km"] = df["distance_km"].fillna(median_distance)

print(f"\nSTEP B: Median age = {median_age}")
print(f"STEP B: Median distance_km = {median_distance}")

# STEP C - Detect and cap outliers separately for distance_km and fee_inr
def cap_outliers_iqr(series):
    """Compute IQR fences, count values outside fences, and clip with np.clip."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    capped_count = int(((series < lower_fence) | (series > upper_fence)).sum())
    capped_series = np.clip(series, lower_fence, upper_fence)
    return capped_series, {
        "q1": q1,
        "q3": q3,
        "iqr": iqr,
        "lower_fence": lower_fence,
        "upper_fence": upper_fence,
        "capped_count": capped_count,
    }

df["distance_km"], dist_stats = cap_outliers_iqr(df["distance_km"])
df["fee_inr"], fee_stats = cap_outliers_iqr(df["fee_inr"])

# STEP D - Final rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nSTEP C & D - distance_km stats:")
for k, v in dist_stats.items():
    print(f"  {k}: {v}")
print(f"  final min: {df['distance_km'].min()}, final max: {df['distance_km'].max()}")

print("\nSTEP C & D - fee_inr stats:")
for k, v in fee_stats.items():
    print(f"  {k}: {v}")
print(f"  final min: {df['fee_inr'].min()}, final max: {df['fee_inr'].max()}")

STEP A: Invalid ages replaced with NaN: 5
STEP A: Total missing age values after replacement: 17

STEP B: Median age = 48.0
STEP B: Median distance_km = 8.25

STEP C & D - distance_km stats:
  q1: 4.625
  q3: 13.0
  iqr: 8.375
  lower_fence: -7.9375
  upper_fence: 25.5625
  capped_count: 26
  final min: 0.6, final max: 25.6

STEP C & D - fee_inr stats:
  q1: 500.0
  q3: 787.5
  iqr: 287.5
  lower_fence: 68.75
  upper_fence: 1218.75
  capped_count: 3
  final min: 300, final max: 1219


## Task 1.4 – Features and NumPy Summary (4 Marks)
Engineer `lead_days` and `no_show`, construct the NumPy feature matrix `X`, and compute vectorised NumPy statistics (column means, long-lead mask `> 14` days, and distance z-scores with `ddof=0`).

In [5]:
# STEP A - Feature engineering
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days
df["no_show"] = (df["status"] == "No-Show").astype(int)

no_show_count = int(df["no_show"].sum())
total_records = len(df)
no_show_pct = round(no_show_count / total_records * 100, 1)

print("STEP A:")
print(f"  Minimum lead_days: {df['lead_days'].min()}")
print(f"  Maximum lead_days: {df['lead_days'].max()}")
print(f"  No-shows: {no_show_count}")
print(f"  Total records: {total_records}")
print(f"  No-show percentage: {no_show_pct}%")

# STEP B - Create NumPy feature matrix
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("\nSTEP B: X shape:", X.shape)

# STEP C - Column means
col_means = np.mean(X, axis=0)
print("STEP C: Column means [age, distance_km, fee_inr, lead_days]:", np.round(col_means, 2))

# STEP D - Long-lead bookings (lead_days > 14)
mask = df["lead_days"].to_numpy() > 14
long_lead_count = int(mask.sum())
long_lead_share = round(float(mask.mean()) * 100, 1)
print(f"STEP D: Long-lead bookings (>14 days): count={long_lead_count}, share={long_lead_share}%")

# STEP E - Distance z-scores (population std, ddof=0)
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
z_outliers = int(np.sum(np.abs(z) > 2))
print(f"STEP E: Rows with |distance z-score| > 2: {z_outliers}")

STEP A:
  Minimum lead_days: 1
  Maximum lead_days: 45
  No-shows: 218
  Total records: 990
  No-show percentage: 22.0%

STEP B: X shape: (990, 4)
STEP C: Column means [age, distance_km, fee_inr, lead_days]: [ 48.17   9.5  635.71  11.82]
STEP D: Long-lead bookings (>14 days): count=296, share=29.9%
STEP E: Rows with |distance z-score| > 2: 53


## Task 1.5 – Enrich Clinics Using Live Open-Meteo API (3 Marks)
Fetch `hourly.temperature_2m[0]` (midnight UTC) for all 8 clinics from `https://api.open-meteo.com/v1/forecast` with `timeout=10` and exactly one retry on failure. Save the enriched table as `clinics_enriched.csv`.

In [6]:
import requests

clinics_df = pd.read_csv(CLINICS_PATH)
OPEN_METEO_URL = "https://api.open-meteo.com/v1/forecast"

def fetch_clinic_temp(clinic_name, latitude, longitude):
    """Fetch first hourly temperature_2m from Open-Meteo with 1 retry."""
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }
    last_reason = "Unknown error"
    for attempt in range(2):
        try:
            response = requests.get(OPEN_METEO_URL, params=params, timeout=10)
            if response.status_code != 200:
                last_reason = f"HTTP status {response.status_code}"
                continue
            data = response.json()
            hourly = data.get("hourly")
            if not hourly or "temperature_2m" not in hourly:
                last_reason = "Missing 'hourly' or 'temperature_2m' field"
                continue
            temps = hourly["temperature_2m"]
            if not temps:
                last_reason = "Empty temperature_2m list"
                continue
            return temps[0]
        except requests.exceptions.RequestException as exc:
            last_reason = f"RequestException: {exc}"
    print(f"API failure for {clinic_name}: {last_reason}")
    return None

temp_values = []
for _, row in clinics_df.iterrows():
    temp_c = fetch_clinic_temp(row["clinic_name"], row["latitude"], row["longitude"])
    temp_values.append(temp_c)

clinics_enriched = clinics_df.copy()
clinics_enriched["temp_c"] = temp_values

clinics_enriched_path = os.path.join(PROCESSED_DIR, "clinics_enriched.csv")
clinics_enriched.to_csv(clinics_enriched_path, index=False)

print("Clinics Enriched DataFrame:")
print(clinics_enriched.to_string(index=False))
print("\nShape:", clinics_enriched.shape)
print("temp_c dtype:", clinics_enriched["temp_c"].dtype)
print("Saved to:", clinics_enriched_path)

API failure for Kothrud Family Clinic: HTTP status 403
API failure for Hadapsar Care Centre: HTTP status 403
API failure for Sitabuldi Health Point: HTTP status 403
API failure for Malviya Nagar Clinic: HTTP status 403
API failure for Vaishali Nagar Clinic: HTTP status 403
API failure for Edappally Medical Centre: HTTP status 403
API failure for Vijay Nagar Clinic: HTTP status 403
API failure for Gomti Nagar Care Point: HTTP status 403
Clinics Enriched DataFrame:
clinic_id              clinic_name    city   tier  latitude  longitude temp_c
      C01    Kothrud Family Clinic    Pune  Metro   18.5074    73.8077   None
      C02     Hadapsar Care Centre    Pune  Metro   18.5089    73.9260   None
      C03   Sitabuldi Health Point  Nagpur Tier-2   21.1458    79.0882   None
      C04     Malviya Nagar Clinic  Jaipur Tier-2   26.8553    75.8150   None
      C05    Vaishali Nagar Clinic  Jaipur Tier-2   26.9124    75.7873   None
      C06 Edappally Medical Centre   Kochi Tier-2   10.0261    7

## Task 1.6 – Export Clean Table and Generate SQL Script (4 Marks)
Validate the cleaned appointments DataFrame, sort by `appointment_id`, format dates as `YYYY-MM-DD`, export `appointments_clean.csv` (990 rows × 16 columns), and generate `load_data.sql` with 8 clinic `INSERT`s followed by 990 appointment `INSERT`s.

In [7]:
REQUIRED_COLUMNS = [
    "appointment_id",
    "patient_id",
    "clinic_id",
    "booked_on",
    "appt_date",
    "age",
    "gender",
    "distance_km",
    "fee_inr",
    "booking_channel",
    "visit_type",
    "reminder_sent",
    "prior_no_shows",
    "status",
    "lead_days",
    "no_show",
]

# STEP A - Final validation
is_unique_id = df["appointment_id"].is_unique
has_all_cols = all(col in df.columns for col in REQUIRED_COLUMNS)
missing_counts = df[REQUIRED_COLUMNS].isnull().sum()
total_missing = int(missing_counts.sum())

print("STEP A - Final Validation:")
print("  Total records:", len(df))
print("  Unique appointment_ids:", df["appointment_id"].nunique(), "| is_unique:", is_unique_id)
print("  All 16 required columns exist:", has_all_cols)
print("  Total missing values across 16 columns:", total_missing)

assert is_unique_id, "appointment_id is not unique!"
assert has_all_cols, "Missing required columns!"
assert total_missing == 0, "Missing values remain in required columns!"

# STEP B - Sort by appointment_id
df = df.sort_values("appointment_id").reset_index(drop=True)

# STEP C - Format dates as YYYY-MM-DD text
df["booked_on"] = df["booked_on"].dt.strftime("%Y-%m-%d")
df["appt_date"] = df["appt_date"].dt.strftime("%Y-%m-%d")

# STEP D - Select exact 16 columns and export CSV
appointments_clean = df[REQUIRED_COLUMNS].copy()
appointments_clean_path = os.path.join(PROCESSED_DIR, "appointments_clean.csv")
appointments_clean.to_csv(appointments_clean_path, index=False)
print(f"\nSTEP D: Saved {appointments_clean.shape} to {appointments_clean_path}")

# STEP E - Generate load_data.sql using f-strings and for loops
def sql_str(val):
    """Escape single quotes and wrap string value in SQL single quotes."""
    escaped = str(val).replace("'", "''")
    return f"'{escaped}'"

sql_statements = []

# 1. All 8 clinic INSERT statements first
clinics_for_sql = pd.read_csv(clinics_enriched_path)
for _, r in clinics_for_sql.iterrows():
    temp_sql = "NULL" if pd.isna(r["temp_c"]) else f"{r['temp_c']}"
    stmt = (
        f"INSERT INTO clinics VALUES ("
        f"{sql_str(r['clinic_id'])},"
        f"{sql_str(r['clinic_name'])},"
        f"{sql_str(r['city'])},"
        f"{sql_str(r['tier'])},"
        f"{r['latitude']},"
        f"{r['longitude']},"
        f"{temp_sql}"
        f");"
    )
    sql_statements.append(stmt)

# 2. All 990 appointment INSERT statements second
for _, r in appointments_clean.iterrows():
    stmt = (
        f"INSERT INTO appointments VALUES ("
        f"{sql_str(r['appointment_id'])},"
        f"{sql_str(r['patient_id'])},"
        f"{sql_str(r['clinic_id'])},"
        f"{sql_str(r['booked_on'])},"
        f"{sql_str(r['appt_date'])},"
        f"{int(r['age'])},"
        f"{sql_str(r['gender'])},"
        f"{float(r['distance_km'])},"
        f"{int(r['fee_inr'])},"
        f"{sql_str(r['booking_channel'])},"
        f"{sql_str(r['visit_type'])},"
        f"{sql_str(r['reminder_sent'])},"
        f"{int(r['prior_no_shows'])},"
        f"{sql_str(r['status'])},"
        f"{int(r['lead_days'])},"
        f"{int(r['no_show'])}"
        f");"
    )
    sql_statements.append(stmt)

load_data_path = os.path.join(SQL_DIR, "load_data.sql")
with open(load_data_path, "w", encoding="utf-8") as f:
    for stmt in sql_statements:
        f.write(stmt + "\n")

clinic_inserts = sum(1 for s in sql_statements if s.startswith("INSERT INTO clinics"))
appt_inserts = sum(1 for s in sql_statements if s.startswith("INSERT INTO appointments"))

print(f"\nSTEP E: Generated {load_data_path}")
print(f"  Clinic INSERT statements     : {clinic_inserts}")
print(f"  Appointment INSERT statements: {appt_inserts}")
print(f"  Total INSERT statements      : {len(sql_statements)}")

STEP A - Final Validation:
  Total records: 990
  Unique appointment_ids: 990 | is_unique: True
  All 16 required columns exist: True
  Total missing values across 16 columns: 0

STEP D: Saved (990, 16) to .\data\processed\appointments_clean.csv

STEP E: Generated .\sql\load_data.sql
  Clinic INSERT statements     : 8
  Appointment INSERT statements: 990
  Total INSERT statements      : 998
